# 04 — TabPFN-TS inference (Prior Labs)

Template to run **TabPFN-TS** over all series. It covers two variants:
1. **Univariate** (target only).
2. **With covariates** (dynamic + static, optional).

**Isolated environment:** `.venv_tabpfn` (see `requirements-tabpfn.txt`).

## 1. Configuration

In [ ]:
from pathlib import Path
import sys
import time
import numpy as np
import pandas as pd
import torch

REPO_ROOT = Path('..').resolve()
sys.path.insert(0, str(REPO_ROOT))

from src.data_loader import load_parquet, filter_period, build_series
from src.metrics import all_metrics

PARQUET_PATH = REPO_ROOT / 'data' / 'anonymized_series.parquet'
OUTPUT_DIR = REPO_ROOT / 'outputs' / 'tabpfn'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

HORIZON = 12

## 2. Loading the series

In [ ]:
df = load_parquet(PARQUET_PATH)
df = filter_period(df, 2020, 2024)
series = build_series(df, min_months=24)
print(f'Series: {len(series)}')

## 3. Loading the TabPFN-TS predictor

In [ ]:
from tabpfn_time_series import (
    TabPFNTimeSeriesPredictor,
    TabPFNMode,
    TimeSeriesDataFrame,
)

predictor = TabPFNTimeSeriesPredictor(tabpfn_mode=TabPFNMode.LOCAL)

## 4. Helper: convert our series to a TimeSeriesDataFrame

In [ ]:
def series_to_tsdf(s, horizon: int):
    train_dates = s.dates[:-horizon]
    test_dates = s.dates[-horizon:]
    train_vals = s.values[:-horizon]
    test_vals = s.values[-horizon:]

    df_train = pd.DataFrame({
        'item_id': s.series_id,
        'date': train_dates,
        'target': train_vals,
    })
    df_test = pd.DataFrame({
        'item_id': s.series_id,
        'date': test_dates,
        'target': np.nan,
    })
    train_tsdf = TimeSeriesDataFrame.from_data_frame(df_train, id_column='item_id', timestamp_column='date')
    test_tsdf = TimeSeriesDataFrame.from_data_frame(df_test, id_column='item_id', timestamp_column='date')
    return train_tsdf, test_tsdf, test_vals

def extract_point(pred_tsdf):
    if 'target' in pred_tsdf.columns:
        return pred_tsdf['target'].values
    cols = [c for c in pred_tsdf.columns if c.startswith('0.5')]
    return pred_tsdf[cols[0]].values

## 5. Mass univariate inference

In [ ]:
pred_rows = []
metric_rows = []
times = []

for sid, s in series.items():
    train_tsdf, test_tsdf, y_true = series_to_tsdf(s, HORIZON)
    train_vec = s.values[:-HORIZON]
    t0 = time.perf_counter()
    pred_tsdf = predictor.predict(train_tsdf, test_tsdf)
    dt = time.perf_counter() - t0
    y_pred = np.clip(extract_point(pred_tsdf)[:HORIZON], 0, None).astype(np.float32)

    m = all_metrics(y_true, y_pred, train_vec)
    metric_rows.append({'series_id': sid, 'model': 'TabPFN_TS', **m})
    times.append({'series_id': sid, 'time_seconds': dt})

    for h in range(HORIZON):
        pred_rows.append({
            'series_id': sid,
            'horizon_month': h + 1,
            'y_true': float(y_true[h]),
            'pred_tabpfn_ts': float(y_pred[h]),
        })

pd.DataFrame(metric_rows).to_csv(OUTPUT_DIR / 'metrics_tabpfn_ts.csv', index=False)
pd.DataFrame(pred_rows).to_csv(OUTPUT_DIR / 'predictions_tabpfn_ts.csv', index=False)
pd.DataFrame(times).to_csv(OUTPUT_DIR / 'inference_times_tabpfn.csv', index=False)

## 6. Covariate variant (optional)

Covariates are added as extra columns to the `TimeSeriesDataFrame`. The structure is analogous to notebook 02, but the columns must accompany the target in each row.

**Finding of the work:** exogenous covariates do not improve significantly over the univariate version (see [docs/05_results.md](../docs/05_results.md)).